In [10]:
from pathlib import Path

import pandas as pd
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

NOTEBOOK_CWD = Path.cwd().resolve()
DATA_FILENAME = "centros_servicios_establecimientos_sanitarios_limpio.csv"

PROJECT_DIR = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD / "analisis_datos", NOTEBOOK_CWD, NOTEBOOK_CWD.parent]
        if (candidate / "data" / "processed" / DATA_FILENAME).exists()
    ),
    NOTEBOOK_CWD,
)

DATA_PATH = PROJECT_DIR / "data" / "processed" / DATA_FILENAME
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, sep=";")


def classify_public(value):
    text = str(value).strip().lower()
    text = (
        text.replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ñ", "n")
    )
    markers = [
        "servicio madrileno de salud",
        "comunidad autonoma",
        "administracion",
        "ministerio",
        "municipal",
        "carlos iii",
    ]
    return any(marker in text for marker in markers)


# Keep only public network.
df = df[df["dependencia_patrimonial"].map(classify_public)].copy()

especialidades_explotadas = (
    df["especialidades_texto"]
    .fillna("")
    .astype(str)
    .str.split(" | ", regex=False)
    .explode()
    .str.strip()
)
especialidades_explotadas = especialidades_explotadas[especialidades_explotadas != ""]

# Filtrar unicamente las especialidades solicitadas para la red de derivacion
especialidades_permitidas = [
    "medicina intensiva", "hemodinámica", "cirugía cardiaca", "neurología",
    "neurocirugía", "cirugía general", "obstetricia", "pediatría",
    "cuidados intensivos neonatales", "quemados", "servicio de transfusión",
    "cirugía general y digestivo"
]
especialidades_explotadas = especialidades_explotadas[
    especialidades_explotadas.str.lower().isin(especialidades_permitidas)
]

resumen_general = pd.DataFrame(
    {
        "metrica": [
            "centros_unicos",
            "especialidades_distintas",
            "centros_publicos",
            "cobertura_geolocalizada_pct",
        ],
        "valor": [
            int(df["centro_id"].nunique()),
            int(especialidades_explotadas.nunique()),
            int(len(df)),
            round(float(df["lat"].notna().mean() * 100), 2),
        ],
    }
)

especialidades_top = (
    especialidades_explotadas.value_counts().rename_axis("especialidad").reset_index(name="centros")
)

distribucion_financiacion = (
    df["dependencia_patrimonial"].value_counts().rename_axis("dependencia_patrimonial").reset_index(name="centros")
)

top_municipios = (
    df["municipio"].value_counts().head(20).rename_axis("municipio").reset_index(name="centros")
)

top_centros_capacidad = (
    df[[
        "centro_id",
        "centro_tipo",
        "municipio",
        "dependencia_patrimonial",
        "direccion_completa",
        "num_especialidades",
        "perfiles_atencion",
    ]]
    .sort_values(["num_especialidades", "municipio"], ascending=[False, True])
    .head(25)
    .reset_index(drop=True)
)

perfiles_explotados = (
    df["perfiles_atencion"]
    .fillna("")
    .astype(str)
    .str.split(" | ", regex=False)
    .explode()
    .str.strip()
)
perfiles_explotados = perfiles_explotados[perfiles_explotados != ""]
perfiles_top = perfiles_explotados.value_counts().rename_axis("perfil").reset_index(name="centros")

intensidad_municipio = (
    df.groupby("municipio", as_index=False)
    .agg(
        centros=("centro_id", "nunique"),
        media_especialidades=("num_especialidades", "mean"),
        max_especialidades=("num_especialidades", "max"),
    )
    .sort_values(["media_especialidades", "centros"], ascending=[False, False])
)
intensidad_municipio["media_especialidades"] = intensidad_municipio["media_especialidades"].round(2)

especialidades_rare = especialidades_top.sort_values("centros", ascending=True).head(20).reset_index(drop=True)

completitud = pd.DataFrame(
    {
        "columna": ["lat", "lon", "direccion_completa", "especialidades_texto", "perfiles_atencion"],
        "nulos": [
            int(df["lat"].isna().sum()),
            int(df["lon"].isna().sum()),
            int(df["direccion_completa"].isna().sum()),
            int(df["especialidades_texto"].isna().sum()),
            int(df["perfiles_atencion"].isna().sum()),
        ],
        "porcentaje_nulo": [
            round(float(df["lat"].isna().mean() * 100), 2),
            round(float(df["lon"].isna().mean() * 100), 2),
            round(float(df["direccion_completa"].isna().mean() * 100), 2),
            round(float(df["especialidades_texto"].isna().mean() * 100), 2),
            round(float(df["perfiles_atencion"].isna().mean() * 100), 2),
        ],
    }
)

# Export CSV artifacts.
resumen_general.to_csv(REPORTS_DIR / "resumen_general.csv", index=False)
especialidades_top.to_csv(REPORTS_DIR / "especialidades_top.csv", index=False)
distribucion_financiacion.to_csv(REPORTS_DIR / "distribucion_financiacion.csv", index=False)
top_municipios.to_csv(REPORTS_DIR / "top_municipios.csv", index=False)
completitud.to_csv(REPORTS_DIR / "completitud_datos.csv", index=False)
top_centros_capacidad.to_csv(REPORTS_DIR / "top_centros_capacidad.csv", index=False)
perfiles_top.to_csv(REPORTS_DIR / "perfiles_top.csv", index=False)
intensidad_municipio.to_csv(REPORTS_DIR / "intensidad_municipio.csv", index=False)

# Build PDF report.
pdf_path = REPORTS_DIR / "reporte_centros_sanitarios_publicos.pdf"
summary_table = [["Metrica", "Valor"]] + [
    [row["metrica"], str(row["valor"])] for _, row in resumen_general.iterrows()
]
spec_table = [["Especialidad", "Centros"]] + [
    [row["especialidad"], str(int(row["centros"]))] for _, row in especialidades_top.head(15).iterrows()
]
mun_table = [["Municipio", "Centros"]] + [
    [row["municipio"], str(int(row["centros"]))] for _, row in top_municipios.head(15).iterrows()
]

styles = getSampleStyleSheet()
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4, leftMargin=36, rightMargin=36, topMargin=36, bottomMargin=36)
story = []
story.append(Paragraph("Reporte de Centros Sanitarios Publicos", styles["Title"]))
story.append(Spacer(1, 8))
story.append(Paragraph("Generado automaticamente desde el CSV procesado (solo red publica).", styles["BodyText"]))
story.append(Spacer(1, 12))

story.append(Paragraph("1) KPIs principales", styles["Heading2"]))
t1 = Table(summary_table, colWidths=[280, 140])
t1.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#0b5cab")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.lightgrey]),
]))
story.append(t1)
story.append(Spacer(1, 14))

story.append(Paragraph("2) Top 15 especialidades", styles["Heading2"]))
t2 = Table(spec_table, colWidths=[320, 100])
t2.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2e7d32")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.beige]),
]))
story.append(t2)
story.append(Spacer(1, 14))

story.append(Paragraph("3) Top 15 municipios", styles["Heading2"]))
t3 = Table(mun_table, colWidths=[320, 100])
t3.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#6a1b9a")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.lavender]),
]))
story.append(t3)

doc.build(story)

print("OK: reporte regenerado con filtro de especialidades especificas")
print(f"Filas públicas analizadas: {len(df):,}")
print(f"PDF: {pdf_path}")
print("CSV exportados:")
for p in sorted(REPORTS_DIR.glob("*.csv")):
    print(f"- {p.name}")

resumen_general

OK: reporte regenerado con filtro de especialidades especificas
Filas públicas analizadas: 27
PDF: C:\Users\User\Desktop\Hackanuevo\Hackathon-HSIL-2026\analisis_datos\reports\reporte_centros_sanitarios_publicos.pdf
CSV exportados:
- completitud_datos.csv
- distribucion_financiacion.csv
- especialidades_top.csv
- intensidad_municipio.csv
- perfiles_top.csv
- resumen_general.csv
- top_centros_capacidad.csv
- top_municipios.csv


,metrica,valor
0,centros_unicos,27.0
1,especialidades_distintas,11.0
2,centros_publicos,27.0
3,cobertura_geolocalizada_pct,96.3
